## Feature Extraction Module

This section implements the feature extraction module as per the provided pipeline, starting with loading the preprocessed data and then demonstrating various vectorization methods.

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
import joblib

# Define the path to the preprocessed dataset
data_path = '/content/drive/MyDrive/22CDS0446_NLP/preprocessed.csv'

# Load the preprocessed dataset
try:
    df = pd.read_csv(data_path)
    print(f"Dataset loaded successfully from {data_path}")
    display(df.head())
except FileNotFoundError:
    print(f"Error: The file at {data_path} was not found. Please ensure the path is correct.")
    df = None


Dataset loaded successfully from /content/drive/MyDrive/22CDS0446_NLP/preprocessed.csv


,id,author,author_channel_id,comment,likes,published_at,parent_id,video_title,original_text_length,processed_text,final_cleaned_text_tokens,final_cleaned_text,cleaned_text_length
0,Ugzw-lexZB2lCxv5b_54AaABAg,@EnzoDavidCivil,UCmuq1L2rtvcSAzC6bWsAmEw,0:30 is like I'm watching superheroes transfor...,0,2026-08-08T21:57:17Z,NaN,David Guetta - Hey Mama (Official Video) ft Ni...,59,is like im watching superheroes transformation,"['like', 'im', 'watching', 'superheroes', 'tra...",like im watching superheroes transformation,43
1,UgxIKesgdE1XvKCdhFB4AaABAg,@ailo8625,UCVkp0qhPUWbfz22l_OfvhBQ,Who ELSE JusT RanDomLY RememBereD Th...,0,2026-08-07T19:23:48Z,NaN,David Guetta - Hey Mama (Official Video) ft Ni...,59,who else just randomly remembered th...,"['else', 'randomly', 'remembered', 'song']",else randomly remembered song,29
2,Ugy2QliQ9RLJKi0T_nZ4AaABAg,@MohamedParvezvenesa,UCM2jlwjXl72t9fauVEoHwVA,My with speak in mobile,0,2026-08-07T18:12:06Z,NaN,David Guetta - Hey Mama (Official Video) ft Ni...,23,my with speak in mobile,"['speak', 'mobile']",speak mobile,12
3,Ugwtxvcy0gjNGorI1ft4AaABAg,@OmerGuzel-wu9xb,UCO4BH2Gx4sxa-rGLGhMasdQ,"I used to listened this song when I was 7, but...",2,2026-08-07T15:44:08Z,NaN,David Guetta - Hey Mama (Official Video) ft Ni...,60,i used to listened this song when i was but n...,"['used', 'listened', 'song', 'im', 'adult']",used listened song im adult,27
4,Ugxjpk_1pWzxxcPQ52F4AaABAg,@xXk1lroY_2002,UCGV5h92c7lbRU7OV93jU6kA,I didn't know that video was influenced by Mad...,0,2026-08-07T04:43:44Z,NaN,David Guetta - Hey Mama (Official Video) ft Ni...,61,i didnt know that video was influenced by mad ...,"['didnt', 'know', 'video', 'influenced', 'mad'...",didnt know video influenced mad max fury road,45


### 1. Split Data

We will split the dataset into training and testing sets. Assuming the preprocessed CSV contains a 'text' column and a 'label' column.

In [3]:
import os

# Define the target directory for saving features (assuming it's defined globally or from a previous cell)
# For robustness, we can re-define it here if not guaranteed to be in scope
output_dir = '/content/drive/MyDrive/22CDS0446_NLP/features/'

if df is not None:
    # Assuming 'final_cleaned_text' is the feature (X)
    if 'final_cleaned_text' in df.columns:
        X = df['final_cleaned_text']
        # *** PLEASE SPECIFY THE COLUMN FOR YOUR LABELS (y) ***
        # For example: y = df['your_label_column_name']
        # For now, I will use a placeholder or assume a dummy if not provided.
        # If you don't have a specific 'label' column yet, please inform me.
        if 'label' in df.columns:
            y = df['label']
        elif 'video_title' in df.columns: # Example assumption if no 'label' column exists directly
            print("Warning: 'label' column not found. Using 'video_title' as a placeholder for target (y). \nPLEASE REPLACE 'video_title' WITH YOUR ACTUAL LABEL COLUMN.")
            y = df['video_title'] # Placeholder: Please replace with actual label column
        else:
            print("Error: No 'label' column found and no clear alternative for target (y). Please specify the label column.")
            y = None # Cannot proceed without a target variable

        if y is not None:
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

            print(f"Data split into training and testing sets.")
            print(f"X_train shape: {X_train.shape}")
            print(f"X_test shape: {X_test.shape}")
            print(f"y_train shape: {y_train.shape}")
            print(f"y_test shape: {y_test.shape}")

            # Save labels for later use in the specified output_dir
            joblib.dump(y_train, os.path.join(output_dir, 'labels_y_train.pkl'))
            joblib.dump(y_test, os.path.join(output_dir, 'labels_y_test.pkl'))
            print(f"Saved labels_y_train.pkl and labels_y_test.pkl to {output_dir}")

    else:
        print("Error: 'final_cleaned_text' column not found in the DataFrame. Please check the preprocessed CSV.")
else:
    print("Data not loaded, skipping split.")

PLEASE REPLACE 'video_title' WITH YOUR ACTUAL LABEL COLUMN.
Data split into training and testing sets.
X_train shape: (90962,)
X_test shape: (22741,)
y_train shape: (90962,)
y_test shape: (22741,)
Saved labels_y_train.pkl and labels_y_test.pkl to /content/drive/MyDrive/22CDS0446_NLP/features/


### 2. Choose Vectorization Method

Here we demonstrate three different methods for text vectorization.

#### Option A: Traditional (TF-IDF / Bag-of-Words)

We will use TF-IDF vectorization as an example of a traditional method.

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib
import os

# Define the target directory for saving features (assuming it's defined globally or from a previous cell)
# For robustness, we can re-define it here if not guaranteed to be in scope
output_dir = '/content/drive/MyDrive/22CDS0446_NLP/features/'

def fit_vectorizer(text_data):
    """Fits a TF-IDF vectorizer to the provided text data."""
    # Ensure all elements are strings and fill any NaN with empty strings
    processed_text_data = text_data.fillna('').astype(str)
    vectorizer = TfidfVectorizer(max_features=5000) # Limiting features for demonstration
    vectorizer.fit(processed_text_data)
    print("TF-IDF Vectorizer fitted.")
    return vectorizer

def transform_text(text_data, vectorizer):
    """Transforms text data using a fitted vectorizer."""
    # Ensure all elements are strings and fill any NaN with empty strings
    processed_text_data = text_data.fillna('').astype(str)
    sparse_matrix = vectorizer.transform(processed_text_data)
    print("Text data transformed into a sparse matrix.")
    return sparse_matrix

if 'X_train' in locals() and 'X_test' in locals():
    # Fit the vectorizer on training data
    tfidf_vectorizer = fit_vectorizer(X_train)

    # Transform training and testing data
    X_train_tfidf = transform_text(X_train, tfidf_vectorizer)
    X_test_tfidf = transform_text(X_test, tfidf_vectorizer)

    print(f"Shape of TF-IDF transformed training data: {X_train_tfidf.shape}")
    print(f"Shape of TF-IDF transformed testing data: {X_test_tfidf.shape}")

    # Save the vectorized data to the specified output_dir
    joblib.dump(X_train_tfidf, os.path.join(output_dir, 'vectors_x_train_tfidf.pkl'))
    joblib.dump(X_test_tfidf, os.path.join(output_dir, 'vectors_x_test_tfidf.pkl'))
    print(f"Saved vectors_x_train_tfidf.pkl and vectors_x_test_tfidf.pkl to {output_dir}")
else:
    print("Training and testing data not available, skipping TF-IDF vectorization. Please ensure data splitting has been performed.")

TF-IDF Vectorizer fitted.
Text data transformed into a sparse matrix.
Text data transformed into a sparse matrix.
Shape of TF-IDF transformed training data: (90962, 5000)
Shape of TF-IDF transformed testing data: (22741, 5000)
Saved vectors_x_train_tfidf.pkl and vectors_x_test_tfidf.pkl to /content/drive/MyDrive/22CDS0446_NLP/features/


#### Option B: Static Embeddings (Word2Vec / GloVe)

For static embeddings, we typically use pre-trained models or train our own. Here's a conceptual example using `gensim` for Word2Vec. You would need to install `gensim` first (`!pip install gensim`).

In [7]:
import numpy as np
import joblib
import os

# Define the target directory for saving features (assuming it's defined globally or from a previous cell)
# For robustness, we can re-define it here if not guaranteed to be in scope
output_dir = '/content/drive/MyDrive/22CDS0446_NLP/features/'

# Install gensim and import Word2Vec to resolve the ModuleNotFoundError
try:
    from gensim.models import Word2Vec
except ModuleNotFoundError:
    print("gensim not found, installing...")
    !pip install gensim
    from gensim.models import Word2Vec # Try importing again after installation

def create_embedding_matrix(word_index, embedding_dim, word2vec_model):
    """Creates an embedding matrix from a Word2Vec model."""
    embedding_matrix = np.zeros((len(word_index) + 1, embedding_dim))
    for word, i in word_index.items():
        try:
            embedding_matrix[i] = word2vec_model.wv[word]
        except KeyError: # Word not in model's vocabulary
            embedding_matrix[i] = np.random.normal(scale=0.6, size=(embedding_dim,))
    print("Embedding matrix created.")
    return embedding_matrix

if 'X_train' in locals():
    # Convert all elements in X_train to string to prevent AttributeError with .split()
    X_train_str = X_train.astype(str)

    # For demonstration, let's create a simple tokenized list from X_train
    # In a real scenario, you'd perform proper tokenization.
    tokenized_sentences = [text.split() for text in X_train_str]

    # Train a simple Word2Vec model (or load a pre-trained one)
    embedding_dim = 100 # Example embedding dimension
    word2vec_model = Word2Vec(sentences=tokenized_sentences, vector_size=embedding_dim, window=5, min_count=1, workers=4)
    print("Word2Vec model trained (or loaded).")

    # Create a word_index (mapping words to integers)
    # This is a simplified example; a Keras Tokenizer would be more robust.
    all_words = [word for sentence in tokenized_sentences for word in sentence]
    unique_words = sorted(list(set(all_words)))
    word_index = {word: i + 1 for i, word in enumerate(unique_words)}

    # Generate embedding matrix
    embedding_matrix = create_embedding_matrix(word_index, embedding_dim, word2vec_model)
    print(f"Shape of embedding matrix: {embedding_matrix.shape}")

    # Save the embedding matrix to the specified output_dir
    joblib.dump(embedding_matrix, os.path.join(output_dir, 'embedding_matrix.pkl'))
    print(f"Saved embedding_matrix.pkl to {output_dir}")
else:
    print("Training data not available, skipping Word2Vec embedding generation. Please ensure data splitting has been performed.")

gensim not found, installing...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 64.8 MB/s eta 0:00:00
Word2Vec model trained (or loaded).
Embedding matrix created.
Shape of embedding matrix: (33399, 100)
Saved embedding_matrix.pkl to /content/drive/MyDrive/22CDS0446_NLP/features/


In [8]:
# Install gensim to resolve the ModuleNotFoundError for Word2Vec
!pip install gensim

#### Option C: Contextual Embeddings (BERT / Hugging Face Tokenizers)

This method uses pre-trained transformer models like BERT from the Hugging Face `transformers` library. You would need to install it first (`!pip install transformers`). This typically involves tokenizing input and then passing it through the model to get embeddings. We will focus on the tokenization aspect here.

In [9]:
# !pip install transformers
from transformers import AutoTokenizer, AutoModel
import torch
import os
import joblib

# Define the target directory for saving features (assuming it's defined globally or from a previous cell)
# For robustness, we can re-define it here if not guaranteed to be in scope
output_dir = '/content/drive/MyDrive/22CDS0446_NLP/features/'

def tokenize_and_encode(texts, tokenizer, max_length=128):
    """Tokenizes and encodes text using a Hugging Face tokenizer."""
    # Ensure all elements are strings before converting to list for the tokenizer
    encoded_input = tokenizer(texts.astype(str).tolist(), padding=True, truncation=True, max_length=max_length, return_tensors='pt')
    print(f"Texts tokenized and encoded. Input IDs shape: {encoded_input['input_ids'].shape}")
    return encoded_input

if 'X_train' in locals() and 'X_test' in locals():
    # Load pre-trained tokenizer (e.g., BERT base uncased)
    tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
    print("BERT Tokenizer loaded.")

    # Tokenize and encode training and testing data
    encoded_train = tokenize_and_encode(X_train, tokenizer)
    encoded_test = tokenize_and_encode(X_test, tokenizer)

    # Convert PyTorch tensors to NumPy arrays and save them in pickle format
    # Save input_ids and attention_mask separately for clarity
    joblib.dump(encoded_train['input_ids'].numpy(), os.path.join(output_dir, 'encoded_train_bert_input_ids.pkl'))
    joblib.dump(encoded_train['attention_mask'].numpy(), os.path.join(output_dir, 'encoded_train_bert_attention_mask.pkl'))
    joblib.dump(encoded_test['input_ids'].numpy(), os.path.join(output_dir, 'encoded_test_bert_input_ids.pkl'))
    joblib.dump(encoded_test['attention_mask'].numpy(), os.path.join(output_dir, 'encoded_test_bert_attention_mask.pkl'))

    print(f"Saved BERT encoded tensors (input_ids and attention_mask) for train and test sets to {output_dir} in pickle format.")

    # Example of how to get actual embeddings (requires the model)
    # model = AutoModel.from_pretrained('bert-base-uncased')
    # with torch.no_grad():
    #     model_output_train = model(**encoded_train)
    #     # Typically use the last hidden state of the [CLS] token for sentence embeddings
    #     train_embeddings = model_output_train.last_hidden_state[:, 0, :].numpy()
    # print(f"Shape of BERT embeddings (training): {train_embeddings.shape}")

else:
    print("Training and testing data not available, skipping BERT tokenization. Please ensure data splitting has been performed.")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

BERT Tokenizer loaded.
Texts tokenized and encoded. Input IDs shape: torch.Size([90962, 128])
Texts tokenized and encoded. Input IDs shape: torch.Size([22741, 128])
Saved BERT encoded tensors (input_ids and attention_mask) for train and test sets to /content/drive/MyDrive/22CDS0446_NLP/features/ in pickle format.


### Output

The `joblib` library is used to save the processed features (`vectors_x.pkl`) and labels (`labels_y.pkl`) for different methods. For BERT, PyTorch tensors are saved.

### Saving Features and Labels

All generated feature vectors and labels will be saved into the specified directory: `/content/drive/MyDrive/22CDS0446_NLP/features/`.

In [10]:
import os

# Define the target directory for saving features
output_dir = '/content/drive/MyDrive/22CDS0446_NLP/features/'

# Create the directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)
print(f"Output directory '{output_dir}' ensured to exist.")


Output directory '/content/drive/MyDrive/22CDS0446_NLP/features/' ensured to exist.
